# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and process the FAIR² Mental Health Survey dataset from Kilifi County (Kenya) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll list all record sets in the metadata and print out their available fields using their `@id` values.

In [ ]:
# List available record sets and their fields
record_sets = []
if hasattr(metadata, 'recordSet'):
    record_sets = metadata.recordSet
else:
    print("No record sets found in metadata.")

# Show record set @ids and contained field @ids
for record_set in record_sets:
    print(f"Record Set @id: {record_set['@id']}")
    # Print field @ids if available
    if 'field' in record_set:
        field_ids = [f["@id"] for f in record_set['field']]
        print("  Fields:")
        for fid in field_ids:
            print(f"    - {fid}")
    if 'column' in record_set:
        column_ids = [c["@id"] for c in record_set['column']]
        print("  Columns:")
        for colid in column_ids:
            print(f"    - {colid}")
    print("\n")
# Save the record set ids for later steps
record_set_ids = [rs['@id'] for rs in record_sets]

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Record sets and field `@id`s are referenced as above.

**Note**: For this demonstration, we use the first available record set if multiple exist.

In [ ]:
# Extract data from all record sets
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for Record Set @id: {record_set_id}")
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head(), "\n")
    else:
        print(f"No records found for Record Set @id: {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Let's apply basic operations such as filtering records, normalizing numeric fields, and grouping by categorical attributes.

Please update `<numeric_field_id>` and `<group_field_id>` based on your data overview above.

In [ ]:
# Choose a record set and field for analysis
# Modify these IDs depending on your dataset overview

# Example: Use first record set and choose fields
if record_set_ids and record_set_ids[0] in dataframes:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    print(f"Working with Record Set @id: {rs_id}")

    # Attempt to guess numeric fields and group fields
    numeric_fields = df.select_dtypes(include='number').columns.tolist()
    print(f"Numeric fields: {numeric_fields}")

    categorical_fields = df.select_dtypes(include='object').columns.tolist()
    print(f"Categorical fields: {categorical_fields}")

    # Use GAD-7 or PHQ-9 score as a numeric field (example field name)
    # Please update to match field @id if needed
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        threshold = df[numeric_field_id].mean()

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical field
        if categorical_fields:
            group_field_id = categorical_fields[0]
            if group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"Grouped data by {group_field_id}, mean of {numeric_field_id}:")
                print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No usable record sets or dataframes found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize a numeric field distribution
if record_set_ids and record_set_ids[0] in dataframes:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    numeric_fields = df.select_dtypes(include='number').columns.tolist()
    if numeric_fields:
        field = numeric_fields[0]
        plt.figure(figsize=(8,5))
        sns.histplot(df[field], bins=20, kde=True)
        plt.title(f"Distribution of {field} (Record Set @id: {rs_id})")
        plt.xlabel(field)
        plt.ylabel("Count")
        plt.show()

    # Plot relationship between two columns
    if len(numeric_fields) > 1:
        plt.figure(figsize=(8,5))
        sns.scatterplot(data=df, x=numeric_fields[0], y=numeric_fields[1])
        plt.title(f"Scatter plot: {numeric_fields[0]} vs {numeric_fields[1]} (Record Set @id: {rs_id})")
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.show()
else:
    print("No numerical fields available for visualization.")

## 6. Conclusion
This notebook demonstrated how to:
- Load a FAIR² Croissant dataset using `mlcroissant`.
- Explore available record sets, fields, and their `@id` references.
- Extract and analyze the tabular survey data.
- Apply simple EDA and visualizations with pandas and seaborn.

Further analysis can use additional field @ids, process missing values, and integrate additional metadata for deeper insights into mental health trends in Kilifi County.